# Session XIII: Organizing Code with Modules

---

## Working with module files from this notebook

Every notebook cell runs as `__main__`, so a `__name__` guard typed in a cell is always true. In the module exercises you create real files instead:

- A cell that starts with `%%writefile name.py` saves the rest of the cell to that file (next to this notebook) instead of running it.
- `!uv run python name.py` runs the file as a script, exactly as in a terminal.
- `import name` loads the file as a module. Python imports a module only once per session: after you change the file, call `importlib.reload(name)` or restart the kernel.

## Exercise 1: Create a Module

Create a module file `text_utils.py` with `clean_text(text)`, `count_words(text)`, and `extract_emails(text)`, plus a `__name__` guard that runs a short demo.

1. Complete the functions in the first cell and run it: `%%writefile` saves the cell to `text_utils.py`.
2. Run the module as a script with `!uv run python text_utils.py`: the demo inside the guard prints.
3. Import the module: the guard is false, so the demo does not run, but the functions are available.

### How to approach this code

The text cleaner normalizes whitespace, the word counter uses the cleaned text, and the regex extracts email-shaped strings. The main guard runs the demo when the file is executed as a script and stays silent when it is imported. Regex checks a pattern, not whether an address exists or can receive mail.

In [ ]:
%%writefile text_utils.py
import re


def clean_text(text: str) -> str:
    # TODO: collapse repeated whitespace and strip the ends
    pass


def count_words(text: str) -> int:
    # TODO: count the words of the cleaned text
    pass


def extract_emails(text: str) -> list[str]:
    # TODO: return every email-shaped string (use re.findall)
    pass


if __name__ == "__main__":
    test = "  Contact  alice@email.com  or bob@work.org   for details  "
    print(clean_text(test))
    print(count_words(test))
    print(extract_emails(test))

In [ ]:
# Run the module as a script: __name__ is "__main__", so the demo prints
!uv run python text_utils.py

In [ ]:
# Import the module: __name__ is "text_utils", so the demo does not run
import importlib

import text_utils

importlib.reload(text_utils)  # picks up your latest version of the file
print(text_utils.clean_text("  too   many    spaces "))
print(text_utils.count_words("one two three"))

---
## Exercise 2: Package Structure

Build an `inventory` package with three modules, then use it from a script:

- `inventory/models.py`: a `Product` class with `name`, `price`, `quantity`, a `total_value` property, and `__repr__`
- `inventory/validators.py`: `validate_price(price)` and `validate_quantity(qty)`, which convert the value and raise `ValueError` when it is invalid
- `inventory/store.py`: an `Inventory` class with `add_product`, `search`, a `total_value` property, and `report`
- `inventory_demo.py`: imports `Inventory`, adds a few products, searches, and prints the report inside a `__name__` guard

Run each `%%writefile` cell, then run the demo script with `!uv run python inventory_demo.py`.

### How to approach this code

The package divides inventory behavior among Product objects, validators, and an Inventory collection. Product owns one item's fields, validation enforces price and quantity rules, and Inventory coordinates search and reports. `store.py` imports from its sibling modules with absolute imports (`from inventory.models import Product`), and `inventory_demo.py` finds the package because the script's folder is first in `sys.path`.

In [ ]:
from pathlib import Path

# A regular package is a folder with an __init__.py file
Path("inventory").mkdir(exist_ok=True)

In [ ]:
%%writefile inventory/__init__.py
"""Inventory package: products, validation rules, and the inventory collection."""

In [ ]:
%%writefile inventory/models.py
class Product:
    # TODO: __init__(name, price, quantity), total_value property, __repr__
    pass

In [ ]:
%%writefile inventory/validators.py
def validate_price(price) -> float:
    # TODO: convert to float and raise ValueError if it is not positive
    pass


def validate_quantity(qty) -> int:
    # TODO: convert to int and raise ValueError if it is negative
    pass

In [ ]:
%%writefile inventory/store.py
from inventory.models import Product
from inventory.validators import validate_price, validate_quantity


class Inventory:
    # TODO: __init__, add_product, search, total_value property, report
    pass

In [ ]:
%%writefile inventory_demo.py
from inventory.store import Inventory


def main() -> None:
    inv = Inventory()
    # TODO: add a few products, search by keyword, and print the report


if __name__ == "__main__":
    main()

In [ ]:
!uv run python inventory_demo.py

---
## Exercise 3: Standard Library

Use `Counter` for the top 3 words of the paragraph, `defaultdict` to group the students by letter grade (A ≥ 90, B ≥ 80, C ≥ 70, D ≥ 60, otherwise F), and `Path` to create a `demo_project` folder with a `main.py` and a `README.md` file inside, then list the file names.

### How to approach this code

`Counter` counts repeated words, `defaultdict(list)` groups records without manual list initialization, and `Path` represents filesystem paths. These standard-library tools replace bookkeeping with operations whose names describe their purpose.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path

paragraph = "Python is great. Python is used for web and data. Python is easy."
students = [("Alice", 92), ("Bob", 78), ("Carol", 95), ("David", 65), ("Eve", 88), ("Frank", 72)]

# TODO: Counter top 3 words
# TODO: defaultdict group students by letter grade
# TODO: Path: create demo_project/ with main.py and README.md, then list the file names

---
## Exercise 4: Refactor Monolithic Code (Capstone)

Mark each function with its target module: models.py, validators.py, calculator.py, reports.py, main.py

### How to approach this code

This exercise separates employee data, validation, reporting, and application flow into modules. When placing code into files, import in one clear direction and keep runnable examples behind the main guard. This organization makes each part independently reusable.


In [ ]:
# MONOLITHIC CODE — Mark each with: # -> MODULE: xxx.py

class Employee:
    def __init__(self, name, role, hourly_rate, hours):
        self.name = name
        self.role = role
        self.hourly_rate = hourly_rate
        self.hours = hours
    def __repr__(self): return f"{self.name} ({self.role})"

def validate_salary(rate):
    if not isinstance(rate, (int, float)) or rate <= 0: raise ValueError(f"Bad rate: {rate}")
    return float(rate)

def validate_hours(hours):
    if not isinstance(hours, (int, float)) or hours < 0 or hours > 80: raise ValueError(f"Bad hours: {hours}")
    return float(hours)

def calc_gross(emp):
    base = emp.hourly_rate * min(emp.hours, 40)
    ot = max(0, emp.hours - 40) * emp.hourly_rate * 1.5
    return round(base + ot, 2)

def calc_tax(gross):
    if gross <= 1000: return round(gross * 0.10, 2)
    elif gross <= 3000: return round(gross * 0.20, 2)
    return round(gross * 0.30, 2)

def payroll_report(employees):
    print("PAYROLL REPORT")
    for emp in employees:
        g = calc_gross(emp)
        t = calc_tax(g)
        print(f"  {emp.name}: Gross ${g:.2f}, Tax ${t:.2f}, Net ${g-t:.2f}")

employees = [
    Employee("Alice", "Dev", validate_salary(55), validate_hours(45)),
    Employee("Bob", "Design", validate_salary(45), validate_hours(40)),
]
payroll_report(employees)

---
## Exercise 5: Tiny Package Design
**Objective:** Split responsibility across a few small modules.

Create a small project folder `score_report/` with three modules:
- `helpers.py` with `normalize_name(name)` for string cleaning
- `reports.py` with `summarize_scores(rows)`, which returns the average and the best student as a readable string
- `main.py`, which imports `summarize_scores`, defines a short list of names and scores, and prints the summary inside a `__name__` guard

Run it with `!uv run python score_report/main.py`. Python puts the script's folder (`score_report/`) first in `sys.path`, which is why `main.py` can write `from reports import summarize_scores`.

### How to approach this code

The name helper trims and title-cases its input. The summary handles an empty collection before computing an average and selecting the highest score with `max(..., key=...)`; the f-string presents the result. Each function lives in its own module, and `main.py` only coordinates them.

In [ ]:
from pathlib import Path

Path("score_report").mkdir(exist_ok=True)

In [ ]:
%%writefile score_report/helpers.py
def normalize_name(name: str) -> str:
    """Remove extra spaces and title-case the name."""
    # TODO: return a cleaned name
    pass

In [ ]:
%%writefile score_report/reports.py
from helpers import normalize_name


def summarize_scores(rows: list[dict]) -> str:
    """Create a simple summary message for score records."""
    # TODO: handle an empty list, compute the average score,
    # find the best student, and return a readable string
    pass

In [ ]:
%%writefile score_report/main.py
from reports import summarize_scores


def main() -> None:
    rows = [{"name": " jane doe ", "score": 92}, {"name": "john", "score": 88}]
    print(summarize_scores(rows))


if __name__ == "__main__":
    main()

In [ ]:
!uv run python score_report/main.py